# Model Evaluation — AI Engineering Interview Prep

Covers: confusion matrix, precision/recall, ROC-AUC, cross-validation, calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]
print("Model ready.")

## 1. Confusion Matrix & Classification Report

In [ ]:
cm = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives (TN):  {tn}  — correctly predicted malignant")
print(f"False Positives (FP): {fp}  — predicted benign, actually malignant")
print(f"False Negatives (FN): {fn}  — predicted malignant, actually benign")
print(f"True Positives (TP):  {tp}  — correctly predicted benign")

print(f"\nPrecision:  TP/(TP+FP) = {tp/(tp+fp):.4f}")
print(f"Recall:     TP/(TP+FN) = {tp/(tp+fn):.4f}")
print(f"F1 Score:   2*P*R/(P+R)= {f1_score(y_te, y_pred):.4f}")
print(f"Specificity:TN/(TN+FP) = {tn/(tn+fp):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax1)
ax1.set_title("Confusion Matrix")

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm.round(3), display_labels=data.target_names).plot(ax=ax2)
ax2.set_title("Normalized Confusion Matrix")
plt.tight_layout()
plt.show()

## 2. ROC-AUC

In [ ]:
models = {
    'Random Forest':     (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'Gradient Boosting': (GradientBoostingClassifier(random_state=42), False),
    'Logistic Reg':      (LogisticRegression(max_iter=200), True),
}
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_tr)

plt.figure(figsize=(8, 6))
for name, (model, scale) in models.items():
    Xtr = scaler.transform(X_tr) if scale else X_tr
    Xte = scaler.transform(X_te) if scale else X_te
    model.fit(Xtr, y_tr)
    probs = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, probs)
    auc = roc_auc_score(y_te, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0,1],[0,1],'k--', label='Random (0.5)')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves'); plt.legend()
plt.show()

## 3. Precision-Recall Curve (imbalanced classes)

In [ ]:
# Simulate imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=20,
                                     weights=[0.95, 0.05], random_state=42)
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X_imb, y_imb, test_size=0.2, stratify=y_imb, random_state=42)
rf_i = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_i.fit(Xtr_i, ytr_i)
probs_i = rf_i.predict_proba(Xte_i)[:, 1]

precision, recall, thresholds = precision_recall_curve(yte_i, probs_i)
ap = average_precision_score(yte_i, probs_i)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, 'b-', lw=2, label=f'AP={ap:.3f}')
plt.axhline(yte_i.mean(), linestyle='--', color='r', label=f'Baseline={yte_i.mean():.3f}')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Imbalanced, 5% positive)')
plt.legend()
plt.show()
print(f"Class imbalance: {yte_i.sum()} positives / {len(yte_i)} total ({yte_i.mean():.1%})")
print("Use PR-AUC/AP when class imbalance is severe — ROC-AUC can be misleadingly high!")

## 4. Cross-Validation & Learning Curves

In [ ]:
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_cv, X, y, cv=skf, scoring='roc_auc')
print(f"5-Fold CV ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"Per-fold: {scores.round(4)}")

# Learning curves
train_sizes, train_scores, val_scores = learning_curve(
    rf_cv, X, y, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10), random_state=42)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(1), 'o-', label='Train')
plt.fill_between(train_sizes,
                 train_scores.mean(1) - train_scores.std(1),
                 train_scores.mean(1) + train_scores.std(1), alpha=0.1)
plt.plot(train_sizes, val_scores.mean(1), 's-', label='Validation')
plt.fill_between(train_sizes,
                 val_scores.mean(1) - val_scores.std(1),
                 val_scores.mean(1) + val_scores.std(1), alpha=0.1)
plt.xlabel('Training Set Size'); plt.ylabel('Accuracy')
plt.title('Learning Curves'); plt.legend()
plt.show()

## Interview Tips

- **F1 vs AUC**: F1 is threshold-dependent; AUC is threshold-free. Use AUC for ranking quality.
- **Imbalanced classes**: use `class_weight='balanced'`, stratified CV, PR-AUC (not ROC-AUC).
- **Threshold tuning**: default 0.5 isn't optimal. Use PR or ROC curve to find business-appropriate threshold.
- **Data leakage in CV**: fit preprocessor INSIDE the fold (use Pipeline), not before CV split.
- **Learning curves**: gap between train/val = variance (overfitting); both low = bias (underfitting).
- **Stratified KFold**: essential for classification — maintains class ratio in each fold.